# Making chloropleth maps in Altair

Here's a quick example of how to make a chloropleth map in Altair.  In this example, we'll work with a fairly large data set of baby names in France from 1900-2019, broken down by department.

To work with geographical data, we'll use the `geopandas`, which loads `pandas` dataframes, but with support for geographical outlines in the `geojson` format.  You can use these dataframes just as you would a regular `pandas` dataframe, but they will include that extra geographical outline data.

To get started, we'll need to import our libraries.

In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

# Reading our names data

Now, let's read in our dataset.  The exported data is in CSV format, but with a `;` separator instead of commas.  The INSEE data collapses rare names or where department-level information has been elided (presumably to protect individuals with uncommon names or who were one of the only ones born with that name in a given year).  We'll strip those out.

In [2]:
names = pd.read_csv("Names_hints/dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
3153947,2,MARTHE,1931,63,24
516389,1,FAYCAL,1990,69,3
862835,1,JIMMY,1985,72,18
1559687,1,TANGUY,2005,78,10
1262436,1,NIELS,2017,74,3


# Loading map data

Next, let's load some map data of regions in France using `geopandas`.  These map data come from the [INSEE] and [IGN] and were processed into the `geojson` format we'll need to work with by [Grégoire David].  Here's the [github] repository.

In this example, we'll work with the simplified departments tiles for the Hexagon, but that repository contains higher-resolution versions, the DOM-TOM, and more.

[Grégoire David]: https://gregoiredavid.fr
[INSEE]: http://www.insee.fr/fr/methodes/nomenclatures/cog/telechargement.asp
[IGN]: https://geoservices.ign.fr/adminexpress
[github]: https://github.com/gregoiredavid/france-geojson/

In [3]:
depts = gpd.read_file('Names_hints/departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
59,59,Nord,"MULTIPOLYGON (((3.0404 50.15971, 3.06301 50.17..."
32,32,Gers,"POLYGON ((0.07605 43.98314, 0.14096 43.99468, ..."
34,34,Hérault,"POLYGON ((3.35836 43.91383, 3.42445 43.9116, 3..."
67,67,Bas-Rhin,"POLYGON ((7.63529 49.05416, 7.67449 49.04504, ..."
44,44,Loire-Atlantique,"POLYGON ((-2.45849 47.44812, -2.45343 47.46207..."


Notice how `depts` is a geopandas dataframe.  We'll use it just as a regular `pandas` dataframe, but it includes the geometry info we need to be able to draw those regions when we pass them into Altair.  We just need to make sure that when we work with our data, we keep them in a geopandas dataframe and not a plain dataframe if we want to draw the departments.

In the next cell, notice how we do a right-merge to bring in department data into names.  We do this as a merge on `depts` because we need a geopandas dataframe.  Remember, `depts` is a geopandas dataframe, while `names` is a regular dataframe.  If we did a left merge on `names`, we'd end up with a regular pandas dataframe. After this merge, both `names` and `depts` will be geopandas dataframes.

**Hint:** Be careful when you do your data joins here.  It's easy to accidentally merge the wrong way to accidentally create a _much bigger_ dataset.

In [4]:
# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
1511138,12,Aveyron,"POLYGON ((2.20748 44.61553, 2.20841 44.64384, ...",1,STEPHAN,1972,12,8
1436883,02,Aisne,"POLYGON ((4.04797 49.40564, 4.03991 49.3974, 4...",1,ROMAIN,1993,02,67
3314665,60,Oise,"POLYGON ((1.78384 49.75831, 1.80898 49.75433, ...",2,OPHÉLIE,1992,60,21
3410989,33,Gironde,"POLYGON ((-0.7188 45.32742, -0.64431 45.32205,...",2,ROSA-MARIA,1971,33,3
82080,59,Nord,"MULTIPOLYGON (((3.0404 50.15971, 3.06301 50.17...",1,ALI,2003,59,20


# Show a name over all years

Now we'll choose a name to show across all years.  To that, we'll group all of the names in a department together (squashing the years together) and use the sum.

In [5]:
grouped = (
    names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False)
         .agg({'nombre': 'sum'})
)
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt') # Add geometry data back in
grouped

,code,nom,geometry,dpt,preusuel,sexe,nombre
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,AARON,1,160
1,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABBY,2,3
2,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDALLAH,1,7
3,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDEL,1,3
4,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDELKADER,1,3
...,...,...,...,...,...,...,...
239574,NaN,NaN,None,974,ÉSAÏE,1,3
239575,NaN,NaN,None,974,ÉTHAN,1,53
239576,NaN,NaN,None,974,ÉTIENNE,1,3
239577,NaN,NaN,None,974,ÉVA,2,32


Now let's pick a name and check out how it's distribution over the last 120 years across Metropolitan France.  In this example, I choose the name “Lucien,” which I rather like for some reason.

In [6]:
name = 'KEVIN'
subset = grouped[grouped.preusuel == name]
alt.Chart(subset).mark_geoshape(stroke='white').encode(
    tooltip=['nom', 'code', 'nombre'],
    color='nombre',
).properties(width=800, height=600)

alt.Chart(...)

In [7]:
import pandas as pd
import altair as alt

# Top 10 prénoms les plus donnés
top_names = (
    names.groupby('preusuel')['nombre']
         .sum()
         .nlargest(10)
         .index
)

time_data = (
    names[names.preusuel.isin(top_names)]
    .groupby(['annais', 'preusuel'], as_index=False)
    ['nombre']
    .sum()
)

alt.Chart(time_data).mark_line().encode(
    x=alt.X('annais:O', title='Année'),
    y=alt.Y('nombre:Q', title='Naissances'),
    color='preusuel:N',
    tooltip=['annais', 'preusuel', 'nombre']
).properties(
    width=800,
    height=400,
    title='Évolution des prénoms les plus populaires'
).interactive()

alt.Chart(...)

In [8]:
fashion_names = ['KEVIN', 'DYLAN', 'ENZO', 'JORDAN', 'BRYAN']

fashion = (
    names[names.preusuel.isin(fashion_names)]
    .groupby(['annais', 'preusuel'], as_index=False)
    ['nombre']
    .sum()
)

alt.Chart(fashion).mark_line().encode(
    x='annais:O',
    y='nombre:Q',
    color='preusuel:N'
).properties(
    width=800,
    height=450,
    title='Exemples de prénoms ayant connu un effet de mode'
)

alt.Chart(...)

In [9]:
regional = (
    names.groupby(['dpt', 'preusuel'], as_index=False)
         .agg({'nombre':'sum'})
)

regional = depts.merge(
    regional,
    how='right',
    left_on='code',
    right_on='dpt'
)

In [10]:
kevin = regional[regional.preusuel == 'KEVIN']

alt.Chart(kevin).mark_geoshape(
    stroke='white'
).encode(
    color=alt.Color(
        'nombre:Q',
        scale=alt.Scale(scheme='reds')
    ),
    tooltip=['nom', 'nombre']
).properties(
    width=700,
    height=600,
    title='Répartition géographique du prénom KEVIN'
)

alt.Chart(...)

In [11]:
interesting = ['KEVIN', 'ENZO', 'YANN', 'NOLWENN']

charts = []

for name in interesting:
    subset = regional[regional.preusuel == name]

    c = (
        alt.Chart(subset)
        .mark_geoshape(stroke='white')
        .encode(
            color='nombre:Q'
        )
        .properties(
            title=name,
            width=250,
            height=200
        )
    )

    charts.append(c)

(charts[0] | charts[1]) & (charts[2] | charts[3])

alt.VConcatChart(...)

In [12]:
mixed = (
    names.groupby('preusuel')['sexe']
         .nunique()
)

mixed = mixed[mixed == 2].index

In [13]:
mixed_popular = (
    names[names.preusuel.isin(mixed)]
    .groupby('preusuel')['nombre']
    .sum()
    .sort_values(ascending=False)
    .head(6)
    .index
)

In [14]:
gender_data = (
    names[names.preusuel.isin(mixed_popular)]
    .groupby(['annais', 'preusuel', 'sexe'], as_index=False)
    ['nombre']
    .sum()
)

gender_data['genre'] = gender_data['sexe'].map({
    1: 'Homme',
    2: 'Femme'
})

alt.Chart(gender_data).mark_line().encode(
    x='annais:O',
    y='nombre:Q',
    color='genre:N',
    strokeDash='genre:N'
).properties(
    width=250,
    height=180
).facet(
    column='preusuel:N'
)

alt.FacetChart(...)

In [15]:
target = "CAMILLE"

gender_data = (
    names[names.preusuel == target]
    .groupby(['annais', 'sexe'], as_index=False)
    ['nombre']
    .sum()
)

gender_data['genre'] = gender_data['sexe'].map({
    1: 'Homme',
    2: 'Femme'
})

In [16]:
alt.Chart(gender_data).mark_line(point=True).encode(
    x=alt.X('annais:O', title='Année'),
    y=alt.Y('nombre:Q', title='Naissances'),
    color=alt.Color('genre:N'),
    tooltip=['annais', 'genre', 'nombre']
).properties(
    width=800,
    height=400,
    title=f'Évolution du prénom {target} selon le sexe'
).interactive()

alt.Chart(...)

In [17]:
births = (
    names.groupby(["annais", "sexe"], as_index=False)
         .agg({"nombre": "sum"})
)

births["genre"] = births["sexe"].map({
    1: "Garçons",
    2: "Filles"
})

alt.Chart(births).mark_line().encode(
    x=alt.X("annais:O", title="Année"),
    y=alt.Y("nombre:Q", title="Nombre de naissances"),
    color=alt.Color("genre:N", title="Sexe"),
    tooltip=["annais", "genre", "nombre"]
).properties(
    width=800,
    height=400,
    title="Évolution du nombre total de naissances par sexe"
)

alt.Chart(...)